# <p style="background-color:skyblue; font-family:'Dancing Script', cursive; font-weight:bold; color:black; font-size:120%; text-align:center; border: 1px solid black; border-radius:10px; padding: 10px; box-shadow: 0 4px 15px rgba(0, 0, 0, 0.2);">Imports Basic </p>

In [1]:
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from copy import deepcopy
import optuna
from scipy.optimize import minimize
import os
import plotly.graph_objs as go 
import seaborn as sns
import matplotlib.pyplot as plt

import re

from tqdm import tqdm
from IPython.display import clear_output
from concurrent.futures import ThreadPoolExecutor

import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

import lightgbm as lgb
from catboost import CatBoostRegressor, CatBoostClassifier, Pool
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import *
from sklearn.preprocessing import *
from sklearn.linear_model import *
from sklearn.metrics import *
from sklearn.impute import *

# <p style="background-color:skyblue; font-family:'Dancing Script', cursive; font-weight:bold; color:black; font-size:120%; text-align:center; border: 1px solid black; border-radius:10px; padding: 10px; box-shadow: 0 4px 15px rgba(0, 0, 0, 0.2);">Load and Preprocess The Data </p>

In [2]:
%%time

train = pd.read_csv('/kaggle/input/playground-series-s4e10/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s4e10/test.csv')
sample = pd.read_csv('/kaggle/input/playground-series-s4e10/sample_submission.csv')
original = pd.read_csv('/kaggle/input/loan-approval-prediction/credit_risk_dataset.csv')

train = train.drop('id',axis=1)
test = test.drop('id',axis=1)

train = pd.concat(objs=[train, original])

train['person_emp_length'] = pd.to_numeric(train['person_emp_length'], errors='coerce')
train['loan_int_rate'] = pd.to_numeric(train['loan_int_rate'], errors='coerce')

cols_to_impute = ['person_emp_length', 'loan_int_rate']

imputer = SimpleImputer(strategy='median')
imputed_values = imputer.fit_transform(train[cols_to_impute])

imputed_df = pd.DataFrame(imputed_values, columns=cols_to_impute, index=train.index)

train[cols_to_impute] = imputed_df

train = train.drop_duplicates()

cat_c = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

def category_c(df):
    global cat_c
    
    for col in cat_c:

        df[col] = df[col].astype('category')
    
    return df

train  = category_c(train)
test   = category_c(test)

def New_fe(df):
    
    df['income_to_loan_ratio'] = df['person_income'] / df['loan_amnt']
    df['age_income_interaction'] = df['person_age'] * df['person_income']
    df['loan_int_rate_squared'] = df['loan_int_rate'] ** 2
    df['emp_length_squared'] = df['person_emp_length'] ** 2
    df['loan_amnt_to_income_ratio'] = df['loan_amnt'] / df['person_income']
    df['loan_int_rate_income_ratio'] = df['loan_int_rate'] / df['person_income']
    df['income_percentage_for_loan'] = df['loan_percent_income'] * 100
    df['monthly_income'] = df['person_income'] / 12
    df['monthly_debt'] = (df['loan_amnt'] + df['loan_int_rate'])
    df['debt_to_income'] = df['monthly_debt'] / df['monthly_income']
    
    return df

train = New_fe(train)
test = New_fe(test)

CPU times: user 398 ms, sys: 75.3 ms, total: 474 ms
Wall time: 628 ms


In [3]:
train.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status,income_to_loan_ratio,age_income_interaction,loan_int_rate_squared,emp_length_squared,loan_amnt_to_income_ratio,loan_int_rate_income_ratio,income_percentage_for_loan,monthly_income,monthly_debt,debt_to_income
0,37,35000,RENT,0.0,EDUCATION,B,6000,11.49,0.17,N,14,0,5.833333,1295000,132.0201,0.0,0.171429,0.000328,17.0,2916.666667,6011.49,2.061082
1,22,56000,OWN,6.0,MEDICAL,C,4000,13.35,0.07,N,2,0,14.000000,1232000,178.2225,36.0,0.071429,0.000238,7.0,4666.666667,4013.35,0.860004
2,29,28800,OWN,8.0,PERSONAL,A,6000,8.90,0.21,N,10,0,4.800000,835200,79.2100,64.0,0.208333,0.000309,21.0,2400.000000,6008.90,2.503708
3,30,70000,RENT,14.0,VENTURE,B,12000,11.11,0.17,N,5,0,5.833333,2100000,123.4321,196.0,0.171429,0.000159,17.0,5833.333333,12011.11,2.059047
4,22,60000,RENT,2.0,MEDICAL,A,6000,6.92,0.10,N,3,0,10.000000,1320000,47.8864,4.0,0.100000,0.000115,10.0,5000.000000,6006.92,1.201384


In [4]:
test.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,income_to_loan_ratio,age_income_interaction,loan_int_rate_squared,emp_length_squared,loan_amnt_to_income_ratio,loan_int_rate_income_ratio,income_percentage_for_loan,monthly_income,monthly_debt,debt_to_income
0,23,69000,RENT,3.0,HOMEIMPROVEMENT,F,25000,15.76,0.36,N,2,2.760000,1587000,248.3776,9.0,0.362319,0.000228,36.0,5750.000000,25015.76,4.350567
1,26,96000,MORTGAGE,6.0,PERSONAL,C,10000,12.68,0.10,Y,4,9.600000,2496000,160.7824,36.0,0.104167,0.000132,10.0,8000.000000,10012.68,1.251585
2,26,30000,RENT,5.0,VENTURE,E,4000,17.19,0.13,Y,2,7.500000,780000,295.4961,25.0,0.133333,0.000573,13.0,2500.000000,4017.19,1.606876
3,33,50000,RENT,4.0,DEBTCONSOLIDATION,A,7000,8.90,0.14,N,7,7.142857,1650000,79.2100,16.0,0.140000,0.000178,14.0,4166.666667,7008.90,1.682136
4,26,102000,MORTGAGE,8.0,HOMEIMPROVEMENT,D,15000,16.32,0.15,Y,4,6.800000,2652000,266.3424,64.0,0.147059,0.000160,15.0,8500.000000,15016.32,1.766626


# <p style="background-color:skyblue; font-family:'Dancing Script', cursive; font-weight:bold; color:black; font-size:120%; text-align:center; border: 1px solid black; border-radius:10px; padding: 10px; box-shadow: 0 4px 15px rgba(0, 0, 0, 0.2);">Visualize The Target and Other Features</p>

In [6]:
%%time

single_plot_distribution('person_home_ownership',train)

CPU times: user 458 ms, sys: 59.4 ms, total: 517 ms
Wall time: 651 ms


In [7]:
%%time

single_plot_distribution('loan_intent',train)

CPU times: user 26.1 ms, sys: 1.18 ms, total: 27.3 ms
Wall time: 25.6 ms


In [8]:
%%time

single_plot_distribution('loan_grade',train)

CPU times: user 25.1 ms, sys: 2.01 ms, total: 27.1 ms
Wall time: 26.2 ms


In [9]:
%%time

single_plot_distribution('loan_status',train)

CPU times: user 25.1 ms, sys: 1.3 ms, total: 26.4 ms
Wall time: 26.5 ms


# <p style="background-color:skyblue; font-family:'Dancing Script', cursive; font-weight:bold; color:black; font-size:120%; text-align:center; border: 1px solid black; border-radius:10px; padding: 10px; box-shadow: 0 4px 15px rgba(0, 0, 0, 0.2);">Encode Data</p>

In [10]:
%%time

def encode_Label(train_df, test_df):
    le_home_ownership = LabelEncoder()
    le_loan_intent = LabelEncoder()
    le_loan_grade = LabelEncoder()
    le_default_on_file = LabelEncoder()
    
    train_df['person_home_ownership'] = le_home_ownership.fit_transform(train_df['person_home_ownership'])
    train_df['loan_intent'] = le_loan_intent.fit_transform(train_df['loan_intent'])
    train_df['loan_grade'] = le_loan_grade.fit_transform(train_df['loan_grade'])
    train_df['cb_person_default_on_file'] = le_default_on_file.fit_transform(train_df['cb_person_default_on_file'])
    
    test_df['person_home_ownership'] = le_home_ownership.transform(test_df['person_home_ownership'])
    test_df['loan_intent'] = le_loan_intent.transform(test_df['loan_intent'])
    test_df['loan_grade'] = le_loan_grade.transform(test_df['loan_grade'])
    test_df['cb_person_default_on_file'] = le_default_on_file.transform(test_df['cb_person_default_on_file'])

    return train_df, test_df

train, test = encode_Label(train, test)
train = train.reset_index(drop=True)
random_row = train.sample(n=1)
train = train.drop(random_row.index)

CPU times: user 169 ms, sys: 23.2 ms, total: 192 ms
Wall time: 192 ms


# <p style="background-color:skyblue; font-family:'Dancing Script', cursive; font-weight:bold; color:black; font-size:120%; text-align:center; border: 1px solid black; border-radius:10px; padding: 10px; box-shadow: 0 4px 15px rgba(0, 0, 0, 0.2);">Model Building</p>

In [11]:
%%time

# Separating features and target variable from the training data
X = train.drop('loan_status', axis=1)
y = train['loan_status']

# SEED and trails
SEED = 42
n_splits = 5

# Getting the column names for both training and test data
feature_columns = X.columns.values

# Converting all feature columns to string datatype for consistency
test[feature_columns] = test[feature_columns].astype(str)
X[feature_columns] = X[feature_columns].astype(str)

def modelBuild(model_template, test_data):
    
    # Initialize StratifiedKFold with shuffling
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    
    # Lists to store AUC scores for each fold
    train_auc_scores = []
    val_auc_scores = []
    
    # Array to store test predictions
    test_predictions = np.zeros((len(test_data), n_splits))

    # Loop through each fold
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # Clone the provided model template to avoid data leakage between folds
        model = clone(model_template)
        
        # Fit the model on the training data
        model.fit(X_train, y_train)

        # Get predicted probabilities for training and validation data
        y_train_pred = model.predict_proba(X_train)[:, 1] 
        y_val_pred = model.predict_proba(X_val)[:, 1]

        # Calculate AUC for training and validation data
        train_auc = roc_auc_score(y_train, y_train_pred)
        val_auc = roc_auc_score(y_val, y_val_pred)
        
        train_auc_scores.append(train_auc)
        val_auc_scores.append(val_auc)

        # Store test predictions for this fold
        test_predictions[:, fold] = model.predict_proba(test_data)[:, 1]

        # Display AUC scores for the current fold
        print(f"Fold {fold+1} - Train AUC: {train_auc:.4f}, Val AUC: {val_auc:.4f}")

    # Display overall average AUC score across all folds
    print(f"Overall Train AUC: {np.mean(train_auc_scores):.4f}")
    print(f"Overall Val AUC: {np.mean(val_auc_scores):.4f}")
    
    # Calculate the mean of test predictions across all folds
    mean_test_preds = np.mean(test_predictions, axis=1)

    # Create a submission DataFrame
    submission = pd.DataFrame({
        'id': sample['id'],
        'loan_status': mean_test_preds
    })

    return submission

CPU times: user 2.23 s, sys: 159 ms, total: 2.39 s
Wall time: 2.38 s


In [12]:
%%time

params = {
    'depth': 8,
    'learning_rate': 0.19793301995319765,
    'bagging_temperature': 0.7779373495258176,
    'l2_leaf_reg': 5,
    'loss_function': 'Logloss',
    'iterations': 300,
    'grow_policy': 'Lossguide',
    'eval_metric': 'AUC',
}

model = CatBoostClassifier(**params, random_seed=SEED, verbose=0,
                                  cat_features=X.columns.values)
submission = modelBuild(model, test)

Fold 1 - Train AUC: 0.9932, Val AUC: 0.9603
Fold 2 - Train AUC: 0.9911, Val AUC: 0.9628
Fold 3 - Train AUC: 0.9915, Val AUC: 0.9635
Fold 4 - Train AUC: 0.9906, Val AUC: 0.9673
Fold 5 - Train AUC: 0.9909, Val AUC: 0.9650
Overall Train AUC: 0.9915
Overall Val AUC: 0.9638
CPU times: user 8min 40s, sys: 27.6 s, total: 9min 8s
Wall time: 2min 26s


# <p style="background-color:skyblue; font-family:'Dancing Script', cursive; font-weight:bold; color:black; font-size:120%; text-align:center; border: 1px solid black; border-radius:10px; padding: 10px; box-shadow: 0 4px 15px rgba(0, 0, 0, 0.2);">Submission</p>

In [13]:
%%time

submission.to_csv('submission.csv', index=False)

submission.head()

CPU times: user 132 ms, sys: 6.94 ms, total: 139 ms
Wall time: 138 ms


,id,loan_status
0,58645,0.995789
1,58646,0.017003
2,58647,0.401267
3,58648,0.006403
4,58649,0.028777
